In [ ]:
#!pip install albumentations
#!pip install segmentation_models_pytorch

In [ ]:
import hashlib
import json
import os
import random
from pathlib import Path

import albumentations as alb
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm


In [ ]:
from src.utils.config import load_config, check_for_version
from src.utils.helpers import init_this_notebook, p  #, data_loader  #, load_images_and_masks

cfg = load_config()
init_this_notebook(cfg.SEED)

version_name, version_path = check_for_version(cfg)
if version_name is None:
    version_name = "v000"
p("Active version", version_name, color = "red", color2 = "blue")

# Adjust checkpoint paths to include version folder
checkpoint_dir = Path(cfg.PATHS_CHECKPOINT).parent / version_name
checkpoint_dir.mkdir(exist_ok = True)
cfg.PATHS_CHECKPOINT = str(checkpoint_dir / "checkpoint.pth")
cfg.PATHS_BEST_MODEL = str(checkpoint_dir / "best_model.pth")

In [ ]:
# ---------------------------------------------------------------
# augmentations and tensor conversion
# ---------------------------------------------------------------

transformer_resize = alb.Compose(
    [alb.Resize(cfg.IMAGE_SIZE, cfg.IMAGE_SIZE), ToTensorV2(), ]
)
# _transform = transforms.Compose([
#     transforms.Resize((256, 256)),
#     transforms.ToTensor(),  # converts to [0,1]
# ])

transformer = alb.Compose(
    [alb.Resize(cfg.IMAGE_SIZE, cfg.IMAGE_SIZE),  #A.LongestMaxSize(max_size=512),
     #A.PadIfNeeded(min_height=512, min_width=512, border_mode=cv2.BORDER_CONSTANT),
     #
     # Geometric
     alb.HorizontalFlip(p = 0.5), alb.VerticalFlip(p = 0.5),
     alb.ShiftScaleRotate(shift_limit = 0.1, scale_limit = 0.1, rotate_limit = 15, p = 0.5),
     alb.Affine(scale = (0.9, 1.1), rotate = (-15, 15), shear = (-10, 10), p = 0.5),  #
     # Color and lighting
     alb.RandomBrightnessContrast(p = 0.5),  ##A.HueSaturationValue(p=0.5),
     alb.CLAHE(p = 0.5),  # adaptive histogram equalization
     ##A.RGBShift(p=5),
     #
     # Noise and blur
     # A.GaussianBlur(p=0.5),
     ## A.MotionBlur(p=1),
     ##A.GaussNoise(p=5),
     #
     # Distortions
     alb.ElasticTransform(0.1), alb.GridDistortion(p = 0.1), alb.OpticalDistortion(p = 0.1),  #
     # Normalization and tensor conversion (always last)
     ### A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
     ToTensorV2(), ]
)



In [ ]:


def load_images_no_masks(image_path, transform = transformer_resize):
    """
    Load images (no masks) for inference.
    Returns a list of (filename, tensor) tuples.
    """
    #image_path = Path(image_path)
    images = [f for f in image_path.glob("*.*") if f.suffix.lower() in [".jpg", ".jpeg", ".png", ".tif", ".tiff"]]

    data = []
    for i, img_path in enumerate(images, 1):

        image = cv2.imread(img_path)
        if image is None:
            p("Failed to read image", img_path, color = "red")
            return None

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if transform:
            img_tensor = transform(image = image)
            image = img_tensor["image"]

        # Make sure tensor is float32 in [0, 1]
        if isinstance(image, torch.Tensor):
            image = image.float()  # ensures float32
            if image.max() > 1.0:
                image /= 255.0

        data.append((img_path.name, image))

    p(f"Loaded", f"{len(data)} images")
    return data


def load_images_and_masks(image_path, mask_path, transform = None):
    images = [f for f in image_path.glob("*.*") if f.suffix.lower() in [".jpg", ".jpeg", ".png", ".tif", ".tiff"]]

    #images = images[:2]
    data = []
    for i, img_path in enumerate(images, 1):

        image = cv2.imread(img_path)
        if image is None:
            p("Failed to read image", img_path, color = "red")
            return

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path / img_path.name, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            p("Failed to read mask", mask_path, color = "orange")
            return

        if transform:
            img_tensor = transform(image = image, mask = mask)
            image = img_tensor["image"]
            mask = img_tensor["mask"]

        data.append((image, mask))

    p(f"Total (with {"no " if transform is None else ""}transformer)", len(data))
    return data

In [ ]:

class FlexibleSegmentationDataset(torch.utils.data.Dataset):

    def __init__(self, data, to_numpy = False, normalize = True, grayscale = False):
        self.data = data
        self.to_numpy = to_numpy
        self.normalize = normalize
        self.grayscale = grayscale

    def __len__(self):
        return len(self.data)

    # HWC = Height × Width × Channels (NumPy and OpenCV)
    # CHW = Channels × Height × Width (required by PyTorch models)
    def __getitem__(self, idx):
        image, mask = self.data[idx]

        # Handle Albumentations output (might be numpy or torch tensor)
        if isinstance(image, torch.Tensor):
            image = image.float()
            # Albumentations ToTensorV2 already gives CHW, but handle grayscale
            if self.grayscale and image.ndim == 2:
                image = image.unsqueeze(0)
        else:
            # Handle NumPy image
            if self.grayscale and image.ndim == 2:
                image = np.expand_dims(image, axis = -1)  # H x W → H x W x 1
            image = torch.tensor(image).permute(2, 0, 1).float()  # HWC → CHW

        # Handle mask (binary or multi-channel)
        if isinstance(mask, torch.Tensor):
            mask = mask.float()
            # Add channel if mask is HxW
            if mask.ndim == 2:
                mask = mask.unsqueeze(0)
        else:
            mask = torch.tensor(mask).float()
            if mask.ndim == 2:
                mask = mask.unsqueeze(0)  # H → 1 x H x W

        # Normalize image to [0, 1] if the data is still in 0–255 range
        if self.normalize and image.max() > 1.0:
            image /= 255.0
            mask /= 255.0

        # Convert to numpy
        if self.to_numpy:
            image_np = image.permute(1, 2, 0).cpu().numpy()  # CHW → HWC
            mask_np = mask.squeeze().cpu().numpy()
            return image_np, mask_np

        return image, mask

    def get_numpy_item(self, idx):
        """Always return NumPy arrays regardless of to_numpy setting."""
        image, mask = self[idx]
        image = image.permute(1, 2, 0).cpu().numpy()
        mask = mask.squeeze().cpu().numpy()
        return image, mask

    @staticmethod
    def hash_image(image):
        """Return MD5 hash for a NumPy array or PyTorch tensor."""
        if isinstance(image, torch.Tensor):
            arr = image.detach().cpu().numpy()
        else:
            arr = np.array(image)
        return hashlib.md5(arr.tobytes()).hexdigest()

#  ==================================================
#  Visualization utilities
#  ==================================================
#
#  def show_image_with_mask(self, index = None, alpha = 0.4):
#      """Display an image and its mask with overlay."""
#      if index is None:
#          index = random.randint(0, len(self) - 1)
#      image, mask = self.get_numpy_item(index)
#      image_disp = (image * 255).astype(np.uint8) if image.max() <= 1.0 else image.astype(np.uint8)
#      mask_disp = (mask * 255).astype(np.uint8) if mask.max() <= 1.0 else mask.astype(np.uint8)
#      mask_rgb = np.zeros_like(image_disp)
#      mask_rgb[:, :, 0] = mask_disp
#      overlay = cv2.addWeighted(image_disp, 1 - alpha, mask_rgb, alpha, 0)
#      plt.figure(figsize = (6, 2), dpi = 100)
#      plt.subplot(1, 2, 1)
#      plt.imshow(image_disp)
#      plt.title(f"Image #{index}")
#      plt.axis("off")
#      plt.subplot(1, 2, 2)
#      plt.imshow(overlay)
#      plt.title("With Mask Overlay")
#      plt.axis("off")
#      plt.tight_layout()
#      plt.show()
#
#  def show_transform_samples(self, index = None, transform = None, num_samples = 3, alpha = 0.4):
#      """Visualize how Albumentations transforms affect an image/mask pair."""
#      if index is None:
#          index = random.randint(0, len(self) - 1)
#      image, mask = self.get_numpy_item(index)
#      image_disp = (image * 255).astype(np.uint8)
#      mask_disp = (mask * 255).astype(np.uint8)
#      self.show_image_with_mask(index = index, alpha = alpha)
#      for _ in range(num_samples - 1):
#          augmented = transform(image = image_disp, mask = mask_disp)
#          aug_img, aug_mask = augmented["image"], augmented["mask"]
#          plt.close("all")
#          self.show_image_with_mask_from_arrays(aug_img, aug_mask, alpha)
#
#  def show_image_with_mask_from_arrays(self, image, mask, alpha = 0.4):
#      """Helper to display given NumPy arrays (no index)."""
#      image_disp = (image * 255).astype(np.uint8) if image.max() <= 1.0 else image.astype(np.uint8)
#      mask_disp = (mask * 255).astype(np.uint8) if mask.max() <= 1.0 else mask.astype(np.uint8)
#      mask_rgb = np.zeros_like(image_disp)
#      mask_rgb[:, :, 0] = mask_disp
#      overlay = cv2.addWeighted(image_disp, 1 - alpha, mask_rgb, alpha, 0)
#      plt.figure(figsize = (6, 2), dpi = 100)
#      plt.subplot(1, 2, 1)
#      plt.imshow(image_disp)
#      plt.title("Original")
#      plt.axis("off")
#      plt.subplot(1, 2, 2)
#      plt.imshow(overlay)
#      plt.title("Overlay")
#      plt.axis("off")
#      plt.tight_layout()
#      plt.show()


In [ ]:
from torch.utils.data import DataLoader, random_split

# ---------------------------------------------------------------
# load images and masks
# ---------------------------------------------------------------

# Load originals
original_data = load_images_and_masks(
    cfg.PATHS_TRAIN_IMAGES, cfg.PATHS_TRAIN_MASKS, transform = transformer_resize
)
original_data = load_images_and_masks(
    cfg.PATHS_TRAIN_IMAGES, cfg.PATHS_TRAIN_MASKS, transform = transformer_resize
)
train_data = []
seen_hashes = set()

# Keep all original images
for image, mask in original_data:
    img_hash = FlexibleSegmentationDataset.hash_image(image)
    train_data.append((image, mask))
    seen_hashes.add(img_hash)

# Generate augmented data
for k in range(1):
    augmented_data = load_images_and_masks(
        cfg.PATHS_TRAIN_IMAGES, cfg.PATHS_TRAIN_MASKS, transform = transformer
    )

    for image, mask in augmented_data:
        img_hash = FlexibleSegmentationDataset.hash_image(image)
        if img_hash not in seen_hashes:
            train_data.append((image, mask))
            seen_hashes.add(img_hash)

# Dataset
train_dataset = FlexibleSegmentationDataset(data = train_data)
p("Total train_dataset records", len(train_dataset))

# Making sure size of all match
for i in range(len(train_dataset)):
    image, mask = train_dataset[i]
    p(f"{i}", f"image {image.shape}, mask {mask.shape}")


In [ ]:

val_size = int(0.2 * len(train_dataset))
train_size = len(train_dataset) - val_size
train_ds, val_ds = random_split(train_dataset, [train_size, val_size])

# DataLoader
p("DataLoader")
train_loader = DataLoader(train_ds, batch_size = cfg.BATCH_SIZE, shuffle = True)
p("Total train_loader  records", len(train_loader), color = "blue")

val_loader = DataLoader(val_ds, batch_size = cfg.BATCH_SIZE, shuffle = True)
p("Total val_loader records", len(val_loader), color = "blue")


In [ ]:
def show_image_with_mask(data, index = 0, alpha = 0.4):
    """
    Display an image and its corresponding mask from an in-memory dataset.
    """
    # Pick random index if needed
    if index == 0:
        index = random.randint(0, len(data) - 1)

    image, mask = data[index]

    # Convert from torch tensor to numpy if needed
    if isinstance(image, torch.Tensor):
        image = image.permute(1, 2, 0).cpu().numpy()
    if isinstance(mask, torch.Tensor):
        mask = mask.cpu().numpy()

    # Handle normalization or scaling
    if image.max() > 1.0:  # probably uint8 0–255
        image_disp = image.astype(np.uint8)
    else:  # probably normalized to 0–1
        image_disp = (image * 255).astype(np.uint8)

    # Normalize mask values to 0–255
    if mask.max() > 1:
        mask_disp = mask.astype(np.uint8)
    else:
        mask_disp = (mask * 255).astype(np.uint8)

    # Create red overlay for mask (in RGB order)
    mask_rgb = np.zeros_like(image_disp)
    mask_rgb[:, :, 0] = mask_disp  # red channel

    # Blend
    overlay = cv2.addWeighted(image_disp, 1 - alpha, mask_rgb, alpha, 0)

    # Plot
    plt.figure(figsize = (6, 3), dpi = 100)
    plt.subplot(1, 2, 1)
    plt.imshow(image)
    if len(data) > 1:
        plt.title(f"Image (index {index})", fontsize = 8)
    else:
        plt.title("Image", fontsize = 8)
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(overlay)
    plt.title("With Mask Overlay", fontsize = 8)
    plt.axis("off")

    plt.tight_layout()
    plt.show()


def show_transform_samples(image, mask, transform, num_samples = 3, alpha = 0.4):
    """
    Apply an Albumentations transform multiple times and display each result
    using the existing show_image_with_mask function.
    """

    # Convert tensors to numpy if needed
    if isinstance(image, torch.Tensor):
        image = image.permute(1, 2, 0).cpu().numpy()
    if isinstance(mask, torch.Tensor):
        mask = mask.cpu().numpy()

    # Convert to uint8 in case image is in [0,1]
    if image.max() <= 1.0:
        image_disp = (image * 255).astype(np.uint8)
    else:
        image_disp = image.astype(np.uint8)

    if mask.max() <= 1.0:
        mask_disp = (mask * 255).astype(np.uint8)
    else:
        mask_disp = mask.astype(np.uint8)

    show_image_with_mask([(image_disp, mask_disp)], index = 0, alpha = alpha)

    # Apply transformations and visualize
    for i in range(num_samples - 1):
        augmented = transform(image = image_disp, mask = mask_disp)
        aug_img, aug_mask = augmented["image"], augmented["mask"]

        # Use your working display function
        show_image_with_mask([(aug_img, aug_mask)], index = 0, alpha = alpha)


for n in range(2):
    num = random.randint(1, 150)
    p(f"index {num}")
    sample_image, sample_mask = train_data[num]
    show_transform_samples(sample_image, sample_mask, transformer, num_samples = 2)




In [ ]:
import segmentation_models_pytorch as smp
import torch

# Model setup
model = smp.Unet(
    encoder_name = "resnet34", encoder_weights = "imagenet", in_channels = 3, classes = 1
    #encoder_name = "Segformer"
    #CLASSES = ["individual_tree", "group_of_trees"]
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = smp.losses.DiceLoss(mode = "binary")
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode = 'min', factor = cfg.SCHEDULER_FACTOR, patience = cfg.SCHEDULER_PATIENCE,
    #verbose = cfg.SCHEDULER_VERBOSE
)
# Only add 'verbose' if the current PyTorch version supports it
# if "verbose" in torch.optim.lr_scheduler.ReduceLROnPlateau.__init__.__code__.co_varnames:
#     scheduler_args["verbose"] = cfg.SCHEDULER_VERBOSE
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, **scheduler_args)

scaler = torch.cuda.amp.GradScaler()

p(model)



In [ ]:



best_val_loss = cfg.BEST_VAL_LOSS
epochs_since_improve = 0
start_epoch = 0

p(f"EPOCHS {cfg.EPOCHS}", color = "blue")

# Pixel Accuracy = (TP + TN) / (TP + TN + FP + FN)
# Dice Coefficient (F1 score) = 2TP / (2TP + FP + FN)
# IoU (Jaccard Index) = TP / (TP + FP + FN)

# Track global performance
global_metrics = {"iou": [], "dice": [], "accuracy": [], "train_loss": [], "val_loss": []}

# --- Try resuming training if checkpoint exists ---
if os.path.exists(cfg.PATHS_CHECKPOINT):
    checkpoint = torch.load(cfg.PATHS_CHECKPOINT, map_location = device)
    version_in_ckpt = checkpoint.get("version", "unknown")
    p(f"Resuming from version: {version_in_ckpt}", color = "cyan")

    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optim_state"])
    scheduler.load_state_dict(checkpoint["sched_state"])
    start_epoch = checkpoint["epoch"] + 1
    best_val_loss = checkpoint.get("best_val_loss", cfg.BEST_VAL_LOSS)
    global_metrics = checkpoint.get("metrics", global_metrics)
    p(f"Resuming from epoch {start_epoch}", color = "salmon")

for epoch in range(start_epoch, cfg.EPOCHS):

    # ------------------- TRAINING -------------------
    model.train()
    total_loss = 0.0

    #for images, masks in train_loader:
    # added tqdm wrapper for training loop
    for images, masks in tqdm(train_loader, desc = f"Epoch {epoch + 1} [Train]", leave = False):
        images = images.to(device, dtype = torch.float32)
        masks = masks.to(device, dtype = torch.float32)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            preds = model(images)
            loss = criterion(preds, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    p(f"Epoch {epoch + 1}", f"Avg Train Loss: {avg_train_loss:.6f}")

    # ------------------- VALIDATION -------------------
    model.eval()
    val_loss = 0.0
    total_iou = 0.0
    total_dice = 0.0
    total_acc = 0.0

    with torch.no_grad():
        #for images, masks in val_loader:
        # added tqdm wrapper for validation loop
        for images, masks in tqdm(val_loader, desc = f"Epoch {epoch + 1} [Val]", leave = False):
            images = images.to(device, dtype = torch.float32)
            masks = masks.to(device, dtype = torch.float32)

            preds = model(images)
            loss = criterion(preds, masks)
            val_loss += loss.item()

            # Compute IoU
            preds_bin = (torch.sigmoid(preds) > 0.5).float()

            # Confusion terms
            tp = (preds_bin * masks).sum()
            fp = (preds_bin * (1 - masks)).sum()
            fn = ((1 - preds_bin) * masks).sum()
            tn = ((1 - preds_bin) * (1 - masks)).sum()

            # Metrics
            iou = (tp + 1e-6) / (tp + fp + fn + 1e-6)
            dice = (2 * tp + 1e-6) / (2 * tp + fp + fn + 1e-6)
            acc = (tp + tn + 1e-6) / (tp + tn + fp + fn + 1e-6)

            total_iou += iou.item()
            total_dice += dice.item()
            total_acc += acc.item()

    avg_val_loss = val_loss / len(val_loader)
    avg_iou = total_iou / len(val_loader)
    avg_dice = total_dice / len(val_loader)
    avg_acc = total_acc / len(val_loader)
    scheduler.step(avg_val_loss)

    # Record global averages
    global_metrics["iou"].append(avg_iou)
    global_metrics["dice"].append(avg_dice)
    global_metrics["accuracy"].append(avg_acc)
    global_metrics["train_loss"].append(avg_train_loss)
    global_metrics["val_loss"].append(avg_val_loss)

    p(
        f"Epoch {epoch + 1}",
        f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | IoU: {avg_iou:.4f} | Dice: {avg_dice:.4f} | Acc: {avg_acc:.4f}"
    )

    # ------------------- CHECKPOINTING -------------------
    is_best = avg_val_loss < best_val_loss
    if is_best:
        best_val_loss = avg_val_loss
        epochs_since_improve = 0
        # Save best model separately
        torch.save(
            {"epoch": epoch, "version": version_name, "model_state": model.state_dict(),
             "optim_state": optimizer.state_dict(), "sched_state": scheduler.state_dict(),
             "best_val_loss": best_val_loss, "metrics": global_metrics, }, cfg.PATHS_BEST_MODEL, )
        p(f"New best model saved at epoch {epoch} with val loss {avg_val_loss:.4f}", color = "green")
    else:
        epochs_since_improve += 1

    # Save checkpoint
    torch.save(
        {"epoch": epoch, "version": version_name, "model_state": model.state_dict(),
         "optim_state": optimizer.state_dict(), "sched_state": scheduler.state_dict(), "best_val_loss": best_val_loss,
         "metrics": global_metrics, }, cfg.PATHS_CHECKPOINT
    )

    # ------------------- EARLY STOPPING -------------------
    if epochs_since_improve >= cfg.EARLY_STOP_PATIENCE:
        p(f"No improvement for {cfg.EARLY_STOP_PATIENCE} epochs", "stopping early.", color = "orange")

        break

# ------------------- FINAL SUMMARY -------------------
p("Training Complete", color = "cyan")

if len(global_metrics["iou"]) > 0:
    mean_iou = float(np.nanmean(global_metrics["iou"]))
    mean_dice = float(np.nanmean(global_metrics["dice"]))
    mean_acc = float(np.nanmean(global_metrics["accuracy"]))
    p(
        "Overall Model Performance", f"IoU: {mean_iou:.4f} | Dice: {mean_dice:.4f} | Accuracy: {mean_acc:.4f}",
        color = "blue"
    )
else:
    p("No validation metrics recorded", "training may have stopped early or validation skipped.", color = "red")



In [ ]:

def predict_dataset(model, dataset, device, num_samples = None, threshold = 0.5):
    """
    Run inference on a dataset and return results for further use or plotting.

    Returns list of dicts ready for plotting or JSON export:
    [
        {
            "index": idx,
            "image": np.ndarray (H, W, 3),
            "mask_true": np.ndarray (H, W),
            "mask_pred": np.ndarray (H, W),
            "iou": float,
            "dice": float
        }, ...
    ]
    """
    model.eval()
    indices = range(len(dataset)) if num_samples is None else random.sample(range(len(dataset)), num_samples)
    results = []

    with torch.no_grad():
        for idx in indices:
            item = dataset[idx]

            # Handle dataset structure
            fname, image, mask = None, None, None
            if isinstance(item, tuple):
                if len(item) == 2 and isinstance(item[0], str):
                    # (filename, tensor)
                    fname, image = item
                elif len(item) == 2:
                    # (image, mask)
                    image, mask = item
                else:
                    raise ValueError(f"Unexpected dataset format at index {idx}: {type(item)} length {len(item)}")
            else:
                image = item

            # Model inference
            image_batch = image.unsqueeze(0).to(device, dtype = torch.float32)
            pred = torch.sigmoid(model(image_batch)).cpu().squeeze().numpy()
            pred_bin = (pred > threshold).astype(np.uint8)

            # Convert to numpy for plotting
            img_np = image.permute(1, 2, 0).cpu().numpy()
            entry = {"index": idx, "file_name": Path(fname).name if fname else f"sample_{idx}.png", "image": img_np,
                     "mask_pred": pred_bin}

            # Optional metrics only if mask exists
            if mask is not None:
                mask_np = mask.cpu().numpy().squeeze()
                mask_bin = (mask_np > 0.5).astype(np.uint8)

                tp = np.sum((mask_bin == 1) & (pred_bin == 1))
                fp = np.sum((mask_bin == 0) & (pred_bin == 1))
                fn = np.sum((mask_bin == 1) & (pred_bin == 0))
                iou = tp / (tp + fp + fn + 1e-6)
                dice = (2 * tp) / (2 * tp + fp + fn + 1e-6)

                entry.update(
                    {"mask_true": mask_bin, "iou": iou, "dice": dice}
                )

            results.append(entry)

    return results


def plot_predictions(results, alpha = 0.5):
    """
    Plot predictions with overlays, given prediction results from predict_dataset().
    """
    for r in results:
        img_disp = (r["image"] * 255).astype(np.uint8) if r["image"].max() <= 1 else r["image"].astype(np.uint8)
        mask_disp = (r["mask_true"] * 255).astype(np.uint8)
        pred_disp = (r["mask_pred"] * 255).astype(np.uint8)

        # Overlays
        mask_rgb = np.zeros_like(img_disp)
        mask_rgb[:, :, 0] = mask_disp
        overlay_truth = cv2.addWeighted(img_disp, 1 - alpha, mask_rgb, alpha, 0)

        pred_rgb = np.zeros_like(img_disp)
        pred_rgb[:, :, 1] = pred_disp
        overlay_pred = cv2.addWeighted(img_disp, 1 - alpha, pred_rgb, alpha, 0)

        missed = ((r["mask_true"] == 1) & (r["mask_pred"] == 0)).astype(np.uint8) * 255
        missed_rgb = np.zeros_like(img_disp)
        missed_rgb[:, :, 0] = missed
        missed_rgb[:, :, 1] = (missed * 0.5).astype(np.uint8)
        overlay_missed = cv2.addWeighted(img_disp, 1 - alpha, missed_rgb, alpha, 0)

        false_new = ((r["mask_pred"] == 1) & (r["mask_true"] == 0)).astype(np.uint8) * 255
        false_rgb = np.zeros_like(img_disp)
        false_rgb[:, :, 0] = false_new
        false_rgb[:, :, 1] = false_new
        overlay_false = cv2.addWeighted(img_disp, 1 - alpha, false_rgb, alpha, 0)

        confusion_map = np.zeros((*r["mask_true"].shape, 3), dtype = np.uint8)
        confusion_map[(r["mask_true"] == 1) & (r["mask_pred"] == 1)] = [255, 255, 255]
        confusion_map[(r["mask_true"] == 0) & (r["mask_pred"] == 1)] = [255, 255, 0]
        confusion_map[(r["mask_true"] == 1) & (r["mask_pred"] == 0)] = [255, 165, 0]

        # Plot for this result
        plt.figure(figsize = (20, 4), dpi = 100)
        plt.subplot(1, 7, 1)
        plt.imshow(img_disp)
        plt.title("Image")
        plt.axis("off")

        plt.subplot(1, 7, 2)
        plt.imshow(overlay_truth)
        plt.title("Truth (Red)")
        plt.axis("off")

        plt.subplot(1, 7, 3)
        plt.imshow(overlay_pred)
        plt.title("Prediction (Green)")
        plt.axis("off")

        plt.subplot(1, 7, 4)
        plt.imshow(overlay_missed)
        plt.title("Missed Truth (Orange)")
        plt.axis("off")

        plt.subplot(1, 7, 5)
        plt.imshow(overlay_false)
        plt.title("False Prediction (Yellow)")
        plt.axis("off")

        plt.subplot(1, 7, 6)
        plt.imshow(confusion_map)
        plt.title(f"Confusion Map\nIoU={r['iou']:.3f}, Dice={r['dice']:.3f}")
        plt.axis("off")

        plt.subplot(1, 7, 7)
        plt.imshow(mask_disp)
        plt.title("Mask")
        plt.axis("off")

        plt.tight_layout()
        plt.show()


#
def show_predictions(name, model, dataset, device, num_samples = 3, alpha = 0.5):
    p(name)
    results = predict_dataset(model, dataset, device, num_samples = num_samples)
    plot_predictions(results, alpha = alpha)


#     """
#     Display a few random predictions from the dataset using a trained model.
#     Includes overlays for:
#     - Ground Truth (Red)
#     - Prediction (Green)
#     - Missed Truth (Blue)
#     - False Prediction (Yellow)
#     - Confusion Matrix Visualization (White=TP, Black=TN, Blue=FN, Yellow=FP)
#     """
#     model.eval()
#     indices = random.sample(range(len(dataset)), num_samples)
#
#     with torch.no_grad():
#         for idx in indices:
#             image, mask = dataset[idx]
#
#             # Prepare for model
#             image_batch = image.unsqueeze(0).to(device, dtype = torch.float32)
#             pred = torch.sigmoid(model(image_batch)).cpu().squeeze().numpy()
#
#             # Convert tensors to numpy
#             img_np = image.permute(1, 2, 0).cpu().numpy() if isinstance(image, torch.Tensor) else image
#             mask_np = mask.cpu().numpy().squeeze() if isinstance(mask, torch.Tensor) else mask.squeeze()
#
#             # Normalize images to uint8
#             img_disp = (img_np * 255).astype(np.uint8) if img_np.max() <= 1.0 else img_np.astype(np.uint8)
#             mask_disp = (mask_np * 255).astype(np.uint8) if mask_np.max() <= 1.0 else mask_np.astype(np.uint8)
#
#             # Prediction binary + scaling
#             pred_bin = (pred > 0.5).astype(np.uint8)
#             pred_disp = (pred_bin * 255).astype(np.uint8)
#
#             # Overlays
#             mask_rgb = np.zeros_like(img_disp)
#             mask_rgb[:, :, 0] = mask_disp  # red for truth
#             overlay_truth = cv2.addWeighted(img_disp, 1 - alpha, mask_rgb, alpha, 0)
#
#             pred_rgb = np.zeros_like(img_disp)
#             pred_rgb[:, :, 1] = pred_disp  # green for prediction
#             overlay_pred = cv2.addWeighted(img_disp, 1 - alpha, pred_rgb, alpha, 0)
#
#             # Missed truth (FN): in truth but not in pred
#             missed = ((mask_np > 0.5) & (pred_bin == 0)).astype(np.uint8) * 255
#             missed_rgb = np.zeros_like(img_disp)
#             missed_rgb[:, :, 0] = missed  # red channel
#             missed_rgb[:, :, 1] = (missed * 0.5).astype(np.uint8)  # green channel (half intensity)
#             overlay_missed = cv2.addWeighted(img_disp, 1 - alpha, missed_rgb, alpha, 0)
#
#             # False predictions (FP): in pred but not in truth
#             false_new = ((pred_bin == 1) & (mask_np <= 0.5)).astype(np.uint8) * 255
#             false_rgb = np.zeros_like(img_disp)
#             false_rgb[:, :, 0] = false_new  # red
#             false_rgb[:, :, 1] = false_new  # green  → yellow = red + green
#             overlay_false = cv2.addWeighted(img_disp, 1 - alpha, false_rgb, alpha, 0)
#
#             # --- NEW: Confusion matrix visualization ---
#             mask_bin = (mask_np > 0.5).astype(np.uint8)
#             confusion_map = np.zeros((*mask_bin.shape, 3), dtype = np.uint8)
#
#             # Ensure confusion_map is 2D before coloring
#             if confusion_map.ndim == 4:
#                 confusion_map = confusion_map.squeeze(0)
#
#             # True Positive = white
#             confusion_map[(mask_bin == 1) & (pred_bin == 1)] = [255, 255, 255]
#             # True Negative = black (default zeros)
#             # False Positive = yellow
#             confusion_map[(mask_bin == 0) & (pred_bin == 1)] = [255, 255, 0]
#             # False Negative = blue
#             confusion_map[(mask_bin == 1) & (pred_bin == 0)] = [0, 0, 255]
#
#             # Metrics
#             tp = np.sum((mask_bin == 1) & (pred_bin == 1))
#             fp = np.sum((mask_bin == 0) & (pred_bin == 1))
#             fn = np.sum((mask_bin == 1) & (pred_bin == 0))
#             iou = tp / (tp + fp + fn + 1e-6)
#             dice = (2 * tp) / (2 * tp + fp + fn + 1e-6)
#
#             # Plot all five
#             plt.figure(figsize = (20, 4), dpi = 100)
#
#             plt.subplot(1, 7, 1)
#             plt.imshow(img_disp)
#             plt.title(f"Image")
#             plt.axis("off")
#
#             plt.subplot(1, 7, 2)
#             plt.imshow(overlay_truth)
#             plt.title(f"Truth (Red)")
#             plt.axis("off")
#
#             plt.subplot(1, 7, 3)
#             plt.imshow(overlay_pred)
#             plt.title("Prediction (Green)")
#             plt.axis("off")
#
#             plt.subplot(1, 7, 4)
#             plt.imshow(overlay_missed)
#             plt.title("Missed Truth (Orange)")
#             plt.axis("off")
#
#             plt.subplot(1, 7, 5)
#             plt.imshow(overlay_false)
#             plt.title("False Prediction (Yellow)")
#             plt.axis("off")
#
#             plt.subplot(1, 7, 6)
#             plt.imshow(confusion_map)
#             plt.title(f"Confusion Map\nIoU={iou:.3f}, Dice={dice:.3f}")
#             plt.axis("off")
#
#             plt.subplot(1, 7, 7)
#             plt.imshow(mask_disp)
#             plt.title(f"Mask")
#             plt.axis("off")
#
#             plt.tight_layout()
#             plt.show()


show_predictions("train_ds", model, train_ds, device, num_samples = 1)
show_predictions("val_ds", model, val_ds, device, num_samples = 1)



In [ ]:
new_images = load_images_no_masks(cfg.PATHS_EVAL_IMAGES, transform = transformer_resize)


In [ ]:
def predict_and_show(model, data, device, alpha = 0.5, num_samples = 1):
    model.eval()
    with torch.no_grad():
        i = 0
        for name, image in data:
            image_input = image.unsqueeze(0).to(device, dtype = torch.float32)
            pred = torch.sigmoid(model(image_input)).cpu().squeeze().numpy()

            # Prepare visualization
            img_np = image.cpu().squeeze().permute(1, 2, 0).numpy()
            img_disp = (img_np * 255).astype(np.uint8)
            pred_bin = (pred > 0.5).astype(np.uint8) * 255

            # Create overlay (green for predicted regions)
            mask_rgb = np.zeros_like(img_disp)
            mask_rgb[:, :, 1] = pred_bin
            overlay = cv2.addWeighted(img_disp, 1 - alpha, mask_rgb, alpha, 0)

            # Plot all three
            plt.figure(figsize = (12, 3), dpi = 100)
            plt.subplot(1, 3, 1)
            plt.imshow(img_disp)
            plt.title(f"Original: {name}")
            plt.axis("off")

            plt.subplot(1, 3, 2)
            plt.imshow(overlay)
            plt.title("Prediction Overlay (Green)")
            plt.axis("off")

            plt.subplot(1, 3, 3)
            plt.imshow(pred_bin)  #, cmap = "gray"
            plt.title("Predicted Mask")
            plt.axis("off")

            plt.tight_layout()
            plt.show()

            i = i + 1
            if i > num_samples:
                break


predict_and_show(model, new_images, device, 3)



In [ ]:
#!pip install rasterio

In [ ]:
## https://zenodo.org/records/11617167
### images.tar.gz
### masks.tar.gz
### train.tar.gz
### test.tar.gz

#mklink /D C:\location C:\source


cfg.ZENODO_PATHS_IMAGES = r"C:\github\Tree-Canopy-Detection\src\data2\train_images"  #"C:\Users\johnh\Downloads\zenodo\images\dataset\holdout\images"
cfg.ZENODO_PATHS_MASKS = r"C:\github\Tree-Canopy-Detection\src\data2\train_masks"  #"C:\Users\johnh\Downloads\zenodo\masks\dataset\holdout\masks"
cfg.ZENODO_PATHS_TEST = r"C:\github\Tree-Canopy-Detection\src\data2\test\test.json"  #"C:\Users\johnh\Downloads\zenodo\test\dataset\holdout"
cfg.ZENODO_PATHS_TRAIN = r"C:\github\Tree-Canopy-Detection\src\data2\train\train.json"  #"C:\Users\johnh\Downloads\zenodo\train\dataset\holdout"

with open(cfg.ZENODO_PATHS_TEST) as f:
    coco = json.load(f)
p("Unique categories:", {c['id']: c['name'] for c in coco['categories']})


In [ ]:



def load_zenodo_dataset(
    image_dir, mask_dir, transform = transformer_resize, samples = 10, json_path = None, allowed_categories = None
):
    image_dir, mask_dir = Path(image_dir), Path(mask_dir)
    images = sorted(image_dir.glob("*.tif"))

    p("Found", f"{len(images)} image files in {image_dir}")

    # --- Load JSON if provided ---
    category_map = {}
    image_filter_set = None

    if json_path and Path(json_path).exists():
        with open(json_path, "r") as f:
            data = json.load(f)

        # Extract all category IDs and names
        if "categories" in data:
            category_map = {c["id"]: c["name"] for c in data["categories"]}
            p("Categories found:", ", ".join(f"{k}:{v}" for k, v in category_map.items()), color = "cyan")

        # If user specified a filter, keep only matching image IDs
        if allowed_categories:
            allowed_categories = set(allowed_categories)
            image_ids = {annotation["image_id"] for annotation in data["annotations"] if
                         annotation["category_id"] in allowed_categories}
            image_filter_set = image_ids
            p(f"Filtering to {len(image_filter_set)} images matching categories {allowed_categories}", color = "blue")

    # --- Normal mask-based loading ---
    data = []
    for img_path in images:
        # Filter out images not in the allowed set (if specified)
        if image_filter_set and Path(json_path).exists():
            # Extract image ID from JSON
            img_id = None
            for img_entry in data.get("images", []):
                if img_entry["file_name"] == img_path.name:
                    img_id = img_entry["id"]
                    break
            if img_id not in image_filter_set:
                continue

        mask_path = Path(mask_dir) / f"{img_path.stem}.png"
        if not mask_path.exists():
            p("Mask missing", f"{img_path.name}", color = "yellow")
            continue

        # Read image
        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # # Read pre-rendered PNG mask
        mask = cv2.imread(str(mask_path))
        mask = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)
        mask = cv2.cvtColor(mask, cv2.COLOR_RGB2GRAY)

        # Convert to binary mask (tree = 255, background = 0)
        mask = ((mask > 0) * 255).astype(np.uint8)

        # print(mask.dtype, mask.min(), mask.max(), np.unique(mask)[:10])
        # print("Total pixels:", mask.size)
        # print("Tree pixels (255):", np.sum(mask == 255))
        # print("Background pixels (0):", np.sum(mask == 0))
        # print("First 200 non-zero mask values:", mask[mask > 0][:200])

        # # Read pre-rendered PNG mask (RGB)
        # mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)
        #
        # if mask is None:
        #     p("Failed to read mask", f"{mask_path.name}", color = "red")
        #     continue
        #
        # # --- Normalize mask to grayscale if needed ---
        # if mask.ndim == 3:
        #     # Convert RGB or RGBA → grayscale safely
        #     if mask.shape[2] == 4:
        #         mask = cv2.cvtColor(mask, cv2.COLOR_BGRA2BGR)
        #     mask_gray = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)
        # else:
        #     mask_gray = mask
        #
        # # --- Ensure mask matches image size ---
        # if mask_gray.shape[:2] != image.shape[:2]:
        #     p("Resizing mismatched mask", f"{mask_path.name} ({mask_gray.shape} → {image.shape[:2]})", color = "orange")
        #     mask_gray = cv2.resize(mask_gray, (image.shape[1], image.shape[0]), interpolation = cv2.INTER_NEAREST)
        #
        # # --- Normalize to binary ---
        # # Some Zenodo masks have many colors; normalize based on unique pixel values
        # unique_vals = np.unique(mask_gray)
        # if len(unique_vals) > 2:
        #     # Use Otsu thresholding to separate tree canopy vs background
        #     _, mask_bin = cv2.threshold(mask_gray, 0, 1, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        # else:
        #     mask_bin = (mask_gray > 0).astype(np.uint8)
        #
        # mask = mask_bin

        # np.set_printoptions(threshold = np.inf)
        # print(mask)
        # np.set_printoptions(threshold = 1000)

        # --- Ensure mask matches image size ---
        # if mask.shape[:2] != image.shape[:2]:
        #     p("Resizing mismatched mask", f"{mask_path.name} ({mask.shape} → {image.shape[:2]})", color = "orange")
        #     mask = cv2.resize(mask, (image.shape[1], image.shape[0]), interpolation = cv2.INTER_NEAREST)

        # plt.figure(figsize = (6, 3), dpi = 100)
        # plt.imshow(mask)
        # plt.title(f"Mask: {mask_path.name}")  #, cmap = "gray"
        # plt.axis("off")
        # plt.show()

        # Apply transformations if provided
        if transform:
            transformed = transform(image = image, mask = mask)
            image, mask = transformed["image"], transformed["mask"]

        data.append((image, mask))

        if len(data) >= samples:
            break

    p(f"Loaded {len(data)} Zenodo samples", color = "green", )
    return data


# zenodo_data = load_images_and_masks(Path(cfg.ZENODO_PATHS_IMAGES), Path(cfg.ZENODO_PATHS_MASKS), transform = transformer_resize)

zenodo_data = load_zenodo_dataset(
    cfg.ZENODO_PATHS_IMAGES, cfg.ZENODO_PATHS_MASKS, json_path = cfg.ZENODO_PATHS_TEST, samples = 50
)

zenodo_ds = FlexibleSegmentationDataset(zenodo_data)

In [ ]:

show_predictions("zenodo_ds", model, zenodo_ds, device, num_samples = 1)
show_predictions("train_ds", model, train_ds, device, num_samples = 1)

In [ ]:

# =========================
from sklearn.metrics import jaccard_score, f1_score, accuracy_score

ious, dices, accs = [], [], []
model.eval()
with torch.no_grad():
    for img, mask in zenodo_ds:
        img = img.unsqueeze(0).to(device)
        pred = torch.sigmoid(model(img)).cpu().squeeze().numpy()
        pred_bin = (pred > 0.5).astype(np.uint8)
        mask_np = mask.squeeze().numpy().astype(np.uint8)

        ious.append(jaccard_score(mask_np.flatten(), pred_bin.flatten()))
        dices.append(f1_score(mask_np.flatten(), pred_bin.flatten()))
        accs.append(accuracy_score(mask_np.flatten(), pred_bin.flatten()))

p(f"Zenodo dataset → IoU={np.mean(ious):.4f}, Dice={np.mean(dices):.4f}, Acc={np.mean(accs):.4f}", color = "cyan")

In [ ]:
p(cfg.PATHS_EVAL_IMAGES)
eval_images = load_images_no_masks(cfg.PATHS_EVAL_IMAGES, transform = transformer_resize)
predict_and_show(model, eval_images, device, 3)
eval_results = predict_dataset(model, eval_images, device)

In [ ]:




def mask_to_polygons(mask):
    """Convert binary mask (0/255 or 0/1) to COCO-style polygons."""
    mask = (mask > 0).astype(np.uint8)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polygons = []
    for cnt in contours:
        cnt = cnt.flatten().tolist()
        if len(cnt) >= 6:  # must be at least 3 points
            polygons.append(cnt)
    return polygons


# def save_predicted_masks(eval_results, save_dir):
#     """
#     Save predicted masks from eval_results into a folder as PNGs.
#     Each entry in eval_results must have 'file_name' and 'mask_pred'.
#     """
#     save_dir = Path(save_dir)
#     save_dir.mkdir(parents = True, exist_ok = True)
#
#     for r in eval_results:
#         file_path = save_dir / f"{Path(r['file_name']).stem}.png"
#         mask_img = (r["mask_pred"] * 255).astype(np.uint8)
#         cv2.imwrite(str(file_path), mask_img)
#
#     print(f"Saved {len(eval_results)} predicted masks to {save_dir}")
#     return save_dir
#
#
# # pred_dir = save_predicted_masks(eval_results, "pred_masks")
# # create_submission(pred_dir, "submission.json")


def export_submission(eval_results, output_dir = "submissions", version_name = version_name):
    """
    Convert eval_results (from predict_dataset) into the official nested submission format:
    {
        "images": [
            {
                "file_name": "...",
                "width": ...,
                "height": ...,
                "cm_resolution": ...,
                "scene_type": "...",
                "annotations": [
                    {
                        "class": "...",
                        "confidence_score": ...,
                        "segmentation": [...]
                    },
                    ...
                ]
            },
            ...
        ]
    }
    """
    os.makedirs(output_dir, exist_ok = True)

    output_path = os.path.join(output_dir, f"submission_{version_name}.json")

    submission = {"images": []}
    grouped = {}

    # Group polygons by image name
    for r in eval_results:
        img_name = Path(r["file_name"]).stem + ".tif"
        mask = (r["mask_pred"] * 255).astype(np.uint8)
        polygons = mask_to_polygons(mask)

        # Each polygon corresponds to one annotation
        anns = []
        for poly in polygons:
            anns.append(
                {"class": r.get("pred_class", "unknown"), "confidence_score": float(r.get("confidence", 0.0)),
                 "segmentation": poly.tolist() if isinstance(poly, np.ndarray) else poly}
            )

        # Collect annotations per image
        if img_name not in grouped:
            grouped[img_name] = anns
        else:
            grouped[img_name].extend(anns)

    # Build final JSON structure
    for img_name, anns in grouped.items():
        submission["images"].append(
            {"file_name": img_name, "width": int(r.get("width", 0)), "height": int(r.get("height", 0)),
             "cm_resolution": int(r.get("cm_resolution", 0)), "scene_type": r.get("scene_type", "unknown"),
             "annotations": anns}
        )

    # Save to file
    with open(output_path, "w") as f:
        json.dump(submission, f, indent = 4)

    p(f"Saved submission for {len(submission['images'])} images to {output_path}")


export_submission(eval_results)

